# Export Breakout Predictions to R2

**Purpose:** Export weekly breakout predictions to Cloudflare R2 for frontend consumption.

**Source:** `main.fantasai.breakout_predictions_current`

**Destination:** R2 bucket at `/fantasai/predictions/breakout_predictions.json`

**Schedule:** Runs after breakout predictions pipeline completes (Tuesday ~10:15 AM ET)

**Output Format:**
```json
{
  "generated_at": "2026-06-01T02:15:23Z",
  "season": 2025,
  "week": 18,
  "predictions": [
    {
      "player_name": "Devontez Walker",
      "position": "WR",
      "team": "BAL",
      "breakout_score": 0.0805,
      "alert_level": "HIGH",
      "snap_share": 0.73,
      "targets": 2,
      "opportunity_score": 1.46
    }
  ]
}
```

**Prerequisites:**
* R2 credentials stored in Databricks secrets (see Setup cell below)
* `boto3` library installed

## R2 Credentials Setup (One-Time)

Before running this notebook, configure R2 credentials using Databricks CLI:

```bash
# Create secret scope (if not exists)
databricks secrets create-scope r2_credentials

# Add R2 credentials
databricks secrets put-secret r2_credentials r2_access_key_id
databricks secrets put-secret r2_credentials r2_secret_access_key
databricks secrets put-secret r2_credentials r2_bucket_name
databricks secrets put-secret r2_credentials r2_endpoint_url  # e.g., https://<account_id>.r2.cloudflarestorage.com
```

Alternatively, use the Databricks UI:
1. Go to **Settings** → **Secrets**
2. Create scope: `r2_credentials`
3. Add the four secrets above

**R2 Endpoint Format:** `https://<account_id>.r2.cloudflarestorage.com`

In [0]:
# Install boto3 for R2 (S3-compatible) uploads
%pip install boto3 --quiet

import boto3
import json
import pandas as pd
from datetime import datetime
from pyspark.sql import functions as F

print("✅ Dependencies loaded")

In [0]:
# Load R2 credentials from Databricks secrets
try:
    r2_access_key = dbutils.secrets.get(scope="r2_credentials", key="r2_access_key_id")
    r2_secret_key = dbutils.secrets.get(scope="r2_credentials", key="r2_secret_access_key")
    r2_bucket = dbutils.secrets.get(scope="r2_credentials", key="r2_bucket_name")
    r2_endpoint = dbutils.secrets.get(scope="r2_credentials", key="r2_endpoint_url")
    
    print("✅ R2 credentials loaded from secrets")
    print(f"   Bucket: {r2_bucket}")
    print(f"   Endpoint: {r2_endpoint}")
except Exception as e:
    print("❌ Failed to load R2 credentials from secrets")
    print(f"   Error: {e}")
    print("\n⚠️  Please configure secrets as described in the setup cell above")
    raise

In [0]:
# Read latest predictions from current table
df = spark.table("main.fantasai.breakout_predictions_current")

print(f"\n📊 Loaded predictions from main.fantasai.breakout_predictions_current")
print(f"   Total players: {df.count()}")

# Get metadata
metadata_row = df.select("season", "week", "generated_at").first()
season = metadata_row.season
week = metadata_row.week
generated_at = metadata_row.generated_at

print(f"   Season: {season}, Week: {week}")
print(f"   Generated: {generated_at}")

# Convert to pandas for JSON serialization
df_export = df.select(
    "player_name",
    "position",
    "team",
    "breakout_score",
    "alert_level",
    "snap_share",
    "snap_share_delta",
    "targets",
    "targets_delta",
    "opportunity_score",
    "fantasy_points",
    "avg_snap_share_prev_2wk",
    "avg_fantasy_points_prev_2wk"
).toPandas()

# Round numeric fields for cleaner JSON
df_export['breakout_score'] = df_export['breakout_score'].round(4)
df_export['snap_share'] = df_export['snap_share'].round(3)
df_export['snap_share_delta'] = df_export['snap_share_delta'].round(3)
df_export['opportunity_score'] = df_export['opportunity_score'].round(2)
df_export['fantasy_points'] = df_export['fantasy_points'].round(1)
df_export['avg_snap_share_prev_2wk'] = df_export['avg_snap_share_prev_2wk'].round(3)
df_export['avg_fantasy_points_prev_2wk'] = df_export['avg_fantasy_points_prev_2wk'].round(1)

# Sort by breakout score descending
df_export = df_export.sort_values('breakout_score', ascending=False)

print(f"✅ Data prepared for export: {len(df_export)} players")

In [0]:
# Build JSON payload
payload = {
    "generated_at": generated_at.isoformat() if generated_at else datetime.now().isoformat(),
    "season": int(season),
    "week": int(week),
    "total_players": len(df_export),
    "high_alerts": int((df_export['breakout_score'] > 0.02).sum()),
    "medium_alerts": int(((df_export['breakout_score'] > 0.01) & (df_export['breakout_score'] <= 0.02)).sum()),
    "predictions": df_export.to_dict('records')
}

# Convert to JSON string
json_data = json.dumps(payload, indent=2)

print("✅ JSON payload created")
print(f"   Size: {len(json_data) / 1024:.1f} KB")
print(f"\nPreview (first 500 chars):")
print(json_data[:500] + "...")

In [0]:
# Initialize S3 client for R2
s3_client = boto3.client(
    's3',
    endpoint_url=r2_endpoint,
    aws_access_key_id=r2_access_key,
    aws_secret_access_key=r2_secret_key,
    region_name='auto'  # R2 uses 'auto' for region
)

# Upload to R2
r2_key = "fantasai/predictions/breakout_predictions.json"

try:
    s3_client.put_object(
        Bucket=r2_bucket,
        Key=r2_key,
        Body=json_data.encode('utf-8'),
        ContentType='application/json',
        CacheControl='max-age=300'  # 5 minute cache
    )
    
    print("\n" + "="*80)
    print("✅ EXPORT COMPLETE")
    print("="*80)
    print(f"\n📤 Uploaded to R2: {r2_key}")
    print(f"   Bucket: {r2_bucket}")
    print(f"   Size: {len(json_data) / 1024:.1f} KB")
    print(f"   Players: {len(df_export)}")
    print(f"   High alerts: {payload['high_alerts']}")
    print(f"   Medium alerts: {payload['medium_alerts']}")
    print(f"\n🌐 Access URL: {r2_endpoint}/{r2_bucket}/{r2_key}")
    print("\n" + "="*80)
    
except Exception as e:
    print("\n❌ Upload failed")
    print(f"   Error: {e}")
    raise

In [0]:
# Verify the file exists in R2
try:
    response = s3_client.head_object(Bucket=r2_bucket, Key=r2_key)
    
    print("\n✅ Upload verification successful")
    print(f"   Last Modified: {response['LastModified']}")
    print(f"   Content Length: {response['ContentLength']} bytes")
    print(f"   Content Type: {response['ContentType']}")
    print(f"   ETag: {response['ETag']}")
    
except Exception as e:
    print("\n⚠️  Verification failed (file may still be uploading)")
    print(f"   Error: {e}")